# Host–Guest Committor: Descriptor Correlation Analysis

Quantifies RPE-weighted physical descriptor correlations and compares them with descriptor relationships learned by the committor model.

This notebook is independently runnable: it imports its dependencies, resolves
the project paths, loads the RPE trainset and trained committor model, creates
the `SystemVisualizer`, and restores or computes the model-output cache.


## 1. Imports and shared analysis helpers


In [ ]:
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import aimmd
import aimmd.pytorch
import aimmd.pytorch.rcmodel
import aimmdTIS
from aimmdTIS import Tools as TAI_Tools
from aimmdTIS.visualization import SystemVisualizer



## 2. Project and model configuration


In [ ]:
DEFAULT_WORK_DIR = Path.cwd()
WORK_DIR = Path(os.environ.get("HG_COMMITTOR_WORK_DIR", DEFAULT_WORK_DIR)).expanduser().resolve()
ANALYSIS_DIR = WORK_DIR

if not ANALYSIS_DIR.is_dir():
    raise FileNotFoundError(
        f"Analysis directory not found: {ANALYSIS_DIR}. "
        "Set HG_COMMITTOR_WORK_DIR to the project root."
    )
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from committor_analysis_helpers import (
    aimmd_softplus_loss_terms,
    conditional_permutation_importance,
    descriptor_names,
    descriptor_swap_loss_matrix,
    permutation_influence_correlation,
    plot_correlated_pair,
    plot_square_matrix,
    predict_q,
    run_descriptor_correlation_tools,
    sigmoid_stable,
    subsample_arrays,
    weighted_gradient_importance,
    weighted_permutation_importance,
)

TRAINSET_FILE = WORK_DIR / "Files" / "trainset_rpe.pkl"
MODEL_DIR = WORK_DIR / "Files"
MODEL_FILE = MODEL_DIR / "aimmd_store_s5em04.h5"
MODEL_KEY = "stage1_full_model_model_best"
OUTPUT_DIR = WORK_DIR / "Figures"
CACHE_DIR = MODEL_DIR / ".cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for required_file in (TRAINSET_FILE, MODEL_FILE):
    if not required_file.is_file():
        raise FileNotFoundError(required_file)

DESCRIPTOR_LABELS = [
    "Host-Guest\n Distance",
    "Guest Orientation\n Angle",
    "Guest Position\n Angle",
    "Hydrogen Bonds",
    "Waters in Cavity",
    "Hydrophobic\n Contact Score",
    "Shared Waters",
]
DESCRIPTOR_UNITS = ["nm", "rad", "rad", "", "", "", ""]
DESC_MIN = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
DESC_MAX = np.array([1.0, np.pi, np.pi, 4.0, 10.0, 60.0, 10.0])
N_BINS_DIM = np.array([200, 200, 200, 6, 50, 200, 50])
STATE_A, STATE_B = 0.05, 1.0
Q_MIN, Q_MAX = -45.0, 15.0
N_DESC = len(DESCRIPTOR_LABELS)


## 3. Load the RPE trainset and committor model


In [ ]:
with TRAINSET_FILE.open("rb") as handle:
    trainset = pickle.load(handle)

required_keys = {"descriptors", "weights", "shot_results"}
missing_keys = required_keys.difference(trainset)
if missing_keys:
    raise KeyError(f"Trainset is missing keys: {sorted(missing_keys)}")

descriptors = np.asarray(trainset["descriptors"])
weights = np.asarray(trainset["weights"], dtype=float).reshape(-1)
shot_results = np.asarray(trainset["shot_results"])

if descriptors.ndim != 2 or descriptors.shape[1] != N_DESC:
    raise ValueError(f"Expected descriptors with shape (N, {N_DESC}), got {descriptors.shape}.")
if len(weights) != len(descriptors) or shot_results.shape != (len(descriptors), 2):
    raise ValueError("Trainset descriptors, weights, and shot results are not aligned.")
if np.any(weights < 0) or not np.all(np.isfinite(weights)):
    raise ValueError("Trainset weights must be finite and non-negative.")

positive_weights = weights[weights > 0]
if positive_weights.size == 0:
    raise ValueError("Trainset contains no positive weights.")
weights = weights / positive_weights.min()
trainset["weights"] = weights

storage = aimmd.Storage(str(MODEL_FILE), mode="r")
if MODEL_KEY not in storage.rcmodels:
    raise KeyError(f"Model key {MODEL_KEY!r} not found; available: {list(storage.rcmodels)}")
model = storage.rcmodels[MODEL_KEY]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TAI_Tools.model_to(model, device)

print(f"Frames: {len(descriptors):,}; descriptors: {descriptors.shape[1]}")
print(f"Weight range: {weights.min():.3g} – {weights.max():.3g}")
print(f"Model: {MODEL_KEY} on {device}")


## 4. Create and initialize the `SystemVisualizer`


In [ ]:
standard_value = np.average(descriptors, axis=0).tolist()
vis = SystemVisualizer(
    temperature=1.0,
    dims_extent=[0.0, 1.2, 0.0, float(np.pi)],
    total_num_descriptors=N_DESC,
    standard_value=standard_value,
    descriptor_labels=DESCRIPTOR_LABELS,
    desc_min=DESC_MIN,
    desc_max=DESC_MAX,
)
vis.load_trainset(trainset)

print(f"Visualizer descriptors: {vis._desc.shape}")
print(f"Visualizer physical descriptors: {vis._desc_phys.shape}")


## 5. Restore or compute model outputs


In [ ]:
Q_CACHE_FILE = CACHE_DIR / "q_model_mbar.npy"
RECOMPUTE_Q_MODEL = False  # set to True after changing the model or trainset
if Q_CACHE_FILE.is_file() and not RECOMPUTE_Q_MODEL:
    q_model = np.load(Q_CACHE_FILE)
    if q_model.shape != (len(descriptors),):
        raise ValueError(
            f"Cached model output has shape {q_model.shape}; expected {(len(descriptors),)}. "
            "Delete the stale cache and rerun this cell."
        )
    p_B = sigmoid_stable(q_model)
    vis._model_output_cache[model] = (p_B, q_model)
    print(f"Loaded model output from {Q_CACHE_FILE}")
else:
    p_B, q_model = vis._model_output_rpe(model)
    np.save(Q_CACHE_FILE, q_model)
    print(f"Computed and cached model output at {Q_CACHE_FILE}")

print(f"q range: {q_model.min():.3f} – {q_model.max():.3f}")


## 6. Prepare a reproducible analysis sample


In [ ]:
valid = np.sum(vis._shot, axis=1) > 0
X_scaled = np.asarray(vis._desc)[valid]
X_physical = np.asarray(vis._desc_phys)[valid]
shots = np.asarray(vis._shot)[valid]
sample_weights = np.asarray(vis._w)[valid]

X_sample, w_sample, shots_sample, sample_indices = subsample_arrays(
    X_scaled,
    weights=sample_weights,
    shots=shots,
    max_samples=250_000,
    random_state=2026,
)
X_physical_sample = X_physical[sample_indices]
names = descriptor_names(N_DESC, DESCRIPTOR_LABELS)
physical_names = descriptor_names(N_DESC, DESCRIPTOR_LABELS, DESCRIPTOR_UNITS)
print(f"Analysis sample: {len(X_sample):,} frames")


## 7. RPE-weighted physical descriptor correlations


In [ ]:
correlation_results = run_descriptor_correlation_tools(
    X_physical,
    descriptor_labels=DESCRIPTOR_LABELS,
    descriptor_units=DESCRIPTOR_UNITS,
    weights=sample_weights,
    max_points=None,
    top_n=15,
)


In [ ]:

fig, ax = plot_square_matrix(
    correlation_results["corr_df"],
    title="RPE-weighted physical descriptor correlation",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    cbar_label="weighted Pearson r",
    fontsize=17
)
ax.set_title("")
TAI_Tools.save_fig_pdf_and_png(fig, "HG_correlation_matrix", output_path=OUTPUT_DIR)
display(correlation_results["top_pairs"])

pair_fig, pair_ax = plot_correlated_pair(correlation_results, pair_rank=0)
TAI_Tools.save_fig_pdf_and_png(pair_fig, "HG_correlation_plot_top_pair", output_path=OUTPUT_DIR)
plt.show()


## 8. Model-space descriptor relationships


In [ ]:
influence_corr, influence_profiles = permutation_influence_correlation(
    model,
    X_scaled,
    shots,
    weights=sample_weights,
    descriptor_names_list=DESCRIPTOR_LABELS,
    n_repeats=3,
)


In [ ]:

fig, ax = plot_square_matrix(
    influence_corr,
    title="Correlation of permutation loss-influence profiles",
    cmap="Reds",
    vmin=0,
    vmax=1,
    cbar_label="Pearson r",
    fontsize=17
)
ax.set_title("")
TAI_Tools.save_fig_pdf_and_png(fig, "HG_permutation_influence_correlation", output_path=OUTPUT_DIR)

swap_loss = descriptor_swap_loss_matrix(
    model,
    X_sample,
    shots_sample,
    weights=w_sample,
    descriptor_names_list=DESCRIPTOR_LABELS,
)
fig, ax = plot_square_matrix(
    swap_loss,
    title="Weighted loss increase after swapping descriptor pairs",
    cmap="magma",
    cbar_label="Δ weighted AIMMD loss",
    fontsize=17
)
TAI_Tools.save_fig_pdf_and_png(fig, "HG_descriptor_swap_loss_matrix", output_path=OUTPUT_DIR)
display(swap_loss)
plt.show()
